In [1]:
import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
import numpy as np

In [2]:
sentence_list = [
    "Meta drops multimodal Llama 3.2 — here's why it's such a big deal",
    "Chip giant Nvidia acquires OctoAI, a Seattle startup that helps companies run AI models",
    "Google is bringing Gemini to all older Pixel Buds",
    "The first Intel Battlmage GPU benchmarks have leaked",
    "Dell partners with Nvidia to accelerate AI adoption in telecoms",
]
ids = ["id1", "id2", "id3", "id4", "id5"]

**Creating a collection**

In [3]:
#chroma_client = chromadb.Client()

In [4]:
# To persist in disk, use:
chroma_client = chromadb.PersistentClient(path="chromadb/")

In [5]:
collection = chroma_client.create_collection(name="udacity")

In [6]:
# By default, Chroma uses the Sentence Transformers all-MiniLM-L6-v2 
# model to create embeddings.
collection.add(
    documents=sentence_list,
    ids=ids
)

In [7]:
collection._embedding_function

In [8]:
collection.count()

5

In [9]:
collection.peek(2)

{'ids': ['id1', 'id2'],
 'embeddings': array([[ 6.06655516e-02, -3.51323076e-02,  6.06437139e-02,
         -5.11926971e-02,  1.13580227e-01, -1.88893322e-02,
         -2.68528406e-02,  5.48635274e-02,  3.23643982e-02,
          5.42442724e-02, -4.04199138e-02, -1.90558545e-02,
         -5.97920641e-02,  2.56031696e-02,  8.48459676e-02,
          4.12196182e-02,  3.95206437e-02, -4.00091298e-02,
         -7.66606405e-02,  2.78291479e-02,  5.38355075e-02,
         -1.35248089e-02,  9.65650082e-02, -3.04361153e-02,
          6.61451789e-03,  7.21731037e-02, -9.53867137e-02,
         -2.75959074e-02,  7.86791742e-03, -6.68519810e-02,
         -1.27341468e-02,  1.21337987e-01, -6.66138530e-02,
         -3.28670405e-02, -6.49284422e-02, -1.61902681e-02,
         -3.32968193e-03,  8.04080814e-02, -3.84503938e-02,
          1.37166106e-04,  3.72595666e-03,  4.83830683e-02,
         -3.69899612e-06, -4.51370515e-02, -1.37449494e-02,
         -7.15254247e-02,  1.01806251e-02, -4.23030294e-02,
  

In [10]:
collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

{'ids': [['id3', 'id1']],
 'embeddings': None,
 'documents': [['Google is bringing Gemini to all older Pixel Buds',
   "Meta drops multimodal Llama 3.2 — here's why it's such a big deal"]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[1.5251754522323608, 1.75485098361969]]}

**Choosing other models**

In [11]:
embeddings_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
embeddings = embeddings_fn(sentence_list)
len(embeddings)

5

In [16]:
print(np.dot(embeddings[1], embeddings[4]))
print(sentence_list[1])
print(sentence_list[4])

0.5583155
Chip giant Nvidia acquires OctoAI, a Seattle startup that helps companies run AI models
Dell partners with Nvidia to accelerate AI adoption in telecoms


In [17]:
from dotenv import load_dotenv
load_dotenv()

embeddings_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [20]:
embeddings_fn.model_name

'text-embedding-ada-002'

In [ ]:
chroma_client.delete_collection(name="udacity")

collection = chroma_client.create_collection(
    name="udacity",
    embedding_function=embeddings_fn
)

In [22]:
collection.add(
    documents=sentence_list,
    ids=ids
)

In [23]:
collection._embedding_function

In [24]:
collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

{'ids': [['id3', 'id4']],
 'embeddings': None,
 'documents': [['Google is bringing Gemini to all older Pixel Buds',
   'The first Intel Battlmage GPU benchmarks have leaked']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.23277103900909424, 0.2440606951713562]]}

**Using with LangChain**

In [25]:
chroma_client.delete_collection(name="udacity")

In [26]:
from dotenv import load_dotenv
load_dotenv()

vector_store = Chroma(
    collection_name="udacity",
    embedding_function=OpenAIEmbeddings(),
)

In [27]:
documents = [
    Document(
        page_content="Meta drops multimodal Llama 3.2 — here's why it's such a big deal",
        metadata={"company":"Meta", "topic": "llama"}
    ),
    Document(
        page_content="Chip giant Nvidia acquires OctoAI, a Seattle startup that helps companies run AI models",
        metadata={"company":"Nvidia", "topic": "acquisition"}
    ),
    Document(
        page_content="Google is bringing Gemini to all older Pixel Buds",
        metadata={"company":"Google", "topic": "gemini"}
    ),
    Document(
        page_content="The first Intel Battlmage GPU benchmarks have leaked",
        metadata={"company":"Intel", "topic": "gpu"}
    ),
    Document(
        page_content="Dell partners with Nvidia to accelerate AI adoption in telecoms",
        metadata={"company":"Dell", "topic": "partnership"}
    ),
]

In [28]:
vector_store.add_documents(documents=documents, ids=ids)

['id1', 'id2', 'id3', 'id4', 'id5']

In [29]:
results = vector_store.similarity_search_with_score(query="gpu",k=2)
for doc, score in results:
    print(f"-> {doc.page_content}\n   [Score={score:.2f}]\n   [{doc.metadata}]\n\n")

-> The first Intel Battlmage GPU benchmarks have leaked
   [Score=0.35]
   [{'company': 'Intel', 'topic': 'gpu'}]


-> Chip giant Nvidia acquires OctoAI, a Seattle startup that helps companies run AI models
   [Score=0.41]
   [{'topic': 'acquisition', 'company': 'Nvidia'}]


